Import

In [2]:
import sys                             # Read system parameters.
import numpy as np                     # Work with multi-dimensional arrays and matrices.
import pandas as pd                    # Manipulate and analyze data.
import matplotlib as mpl               # Create 2D charts.
import matplotlib.pyplot as plt
import seaborn as sb                   # Perform data visualization.
import sklearn                         # Perform data mining and analysis.
from sklearn.model_selection import cross_val_score, StratifiedKFold,train_test_split
from sklearn.svm import SVC
from sklearn import datasets

# Load the dataset.
df = pd.read_csv('../../datasets/group_14.csv')
df["focus_factor"] = (
    df["focus_factor"]
    .astype(str)                  # ensure it's string
    .str.replace(",", ".", regex=False)  # replace comma with dot
)
df["focus_factor"] = pd.to_numeric(df["focus_factor"], errors="coerce").astype(float)
print('Loaded {} records.'.format(len(df)))

Loaded 3000 records.


The dataset

In [3]:
# Split dataset
data_set = df.copy()
y = data_set['target_class']
X = data_set.drop(columns=["target_class"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

Test all possible kernels (without hyperparameter optimization);

Evaluate an SVM model using a holdout test set

In [4]:


def svm_kernel_comparison(X_train, y_train, X_test, y_test, kernels=['linear', 'poly', 'rbf', 'sigmoid'], random_state=1936):
    results = {}
    for kernel in kernels:
        svm = SVC(kernel=kernel, random_state=random_state)
        svm.fit(X_train, np.ravel(y_train))
        train_score = svm.score(X_train, y_train)
        test_score = svm.score(X_test, y_test)
        results[kernel] = {'train_accuracy': train_score, 'test_accuracy': test_score}
        print(f'Kernel: {kernel}')
        print('Train Accuracy: {:.0f}%'.format(train_score * 100))
        print('Test Accuracy: {:.0f}%'.format(test_score * 100))
        print('-' * 30)
        print(svm._gamma)
    return results

results = svm_kernel_comparison(X_train, y_train, X_test, y_test)


Kernel: linear
Train Accuracy: 94%
Test Accuracy: 94%
------------------------------
0.023527795244360192
Kernel: poly
Train Accuracy: 86%
Test Accuracy: 78%
------------------------------
0.023527795244360192
Kernel: rbf
Train Accuracy: 85%
Test Accuracy: 79%
------------------------------
0.023527795244360192
Kernel: sigmoid
Train Accuracy: 66%
Test Accuracy: 66%
------------------------------
0.023527795244360192


Kernel: linear
No overfitting
There is no indication of overfitting, as both accuracies are equal and high. This suggests the model has generalized well to new data

Kernel: poly
There is evidence of overfitting, as the training accuracy is much higher than the test accuracy. The model learned the training patterns very well but did not generalize well to new data.

Kernel: rbf
There is also evidence of overfitting, although less severe than with the poly kernel. The model overfitted the training data but still has some generalization ability.

Kernel: sigmoid
There is no overfitting, but the model may be suffering from underfitting (it did not learn well from either the training or test data)


### SVM without hyperparameter tuning

Using `C = 100` and default hyperparameters, the results on the hold‑out set were:

- Kernel linear: Train 100%, Test 100%
- Kernel poly: Train 99%, Test 80%
- Kernel rbf: Train 99%, Test 82%
- Kernel sigmoid: Train 64%, Test 64%

The linear kernel shows excellent generalization (no overfitting), while poly and rbf present clear signs of overfitting, and sigmoid underfits the data.

In [5]:
svm = SVC(kernel = 'poly', C = 100, random_state = 1936)
svm.fit(X_train, np.ravel(y_train))

# Score using the train data.
train_score = svm.score(X_train, y_train)

# Score using the test data.
test_score = svm.score(X_test, y_test)

print('Train Accuracy: {:.0f}%'.format(train_score * 100))
print('Test Accuracy: {:.0f}%'.format(test_score * 100))


Train Accuracy: 99%
Test Accuracy: 80%


In [6]:
svm = SVC(kernel = 'rbf', C = 100, random_state = 1936)
svm.fit(X_train, np.ravel(y_train))

# Score using the train data.
train_score = svm.score(X_train, y_train)

# Score using the test data.
test_score = svm.score(X_test, y_test)

print('Train Accuracy: {:.0f}%'.format(train_score * 100))
print('Test Accuracy: {:.0f}%'.format(test_score * 100))


Train Accuracy: 99%
Test Accuracy: 82%


In [7]:
svm = SVC(kernel = 'sigmoid', C = 100, random_state = 1936)
svm.fit(X_train, np.ravel(y_train))

# Score using the test data.
score = svm.score(X_test, y_test)

print('Accuracy: {:.0f}%'.format(score * 100))



Accuracy: 64%


Optimize the SVM model with grid search and cross-validation

In [8]:
from sklearn.model_selection import GridSearchCV

svm = SVC(gamma = 'auto', random_state = 1936)

grid = [{'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
         'C': [0.01, 0.1, 1, 5, 10, 25, 50, 100]}]

search = GridSearchCV(svm, param_grid = grid, scoring = 'accuracy', cv = 5)
search.fit(X_train, np.ravel(y_train));

print(search.best_params_)

{'C': 5, 'kernel': 'linear'}


In [9]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.svm import SVC
import numpy as np

param_grid = [
    {
        'kernel': ['linear'],
        'C': [0.01, 0.1, 1, 5, 10, 25, 50, 100]
    },
    {
        'kernel': ['rbf'],
        'C': [0.1, 1, 5, 10, 25, 50, 100],
        'gamma': ['scale', 0.01, 0.1, 1]
    },
    {
        'kernel': ['poly'],
        'C': [0.1, 1, 5, 10, 25, 50, 100],
        'degree': [2, 3],
        'gamma': ['scale', 0.01, 0.1]
    }
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1936)

base_svm = SVC(random_state=1936)

grid_search = GridSearchCV(
    estimator=base_svm,
    param_grid=param_grid,
    scoring='accuracy',
    cv=cv,
    n_jobs=-1
)

grid_search.fit(X_train, np.ravel(y_train))

print("Best parameters:", grid_search.best_params_)
print("Best CV accuracy: {:.2f}%".format(grid_search.best_score_ * 100))

best_svm = grid_search.best_estimator_

train_score = best_svm.score(X_train, y_train)
test_score = best_svm.score(X_test, y_test)

print("Train accuracy (best SVM): {:.2f}%".format(train_score * 100))
print("Test accuracy (best SVM): {:.2f}%".format(test_score * 100))


Best parameters: {'C': 5, 'kernel': 'linear'}
Best CV accuracy: 99.96%
Train accuracy (best SVM): 99.96%
Test accuracy (best SVM): 100.00%


### Hyperparameter tuning and overfitting control

A grid search with 5‑fold stratified cross‑validation was used to tune `C`, `gamma` and, for the polynomial kernel, `degree`. This procedure selects hyperparameters that generalize well across folds, reducing the risk of overfitting to the training set.

The best configuration found was: {'C': 5, 'kernel': 'linear'}, with a mean cross‑validation accuracy of approximately 99.96%. On the hold‑out test set, this model achieved:

- Train accuracy: 99.96%
- Test accuracy: 100.00%

The small gap between train and test accuracy indicates that the tuned SVM does not overfit the training data. In contrast, the untuned polynomial and RBF kernels previously showed large train–test gaps (≈99% vs. 80–82%), which are clear signs of overfitting.


### Best SVM model and comparison with previous methods

The best SVM model is the tuned {'C': 5, 'kernel': 'linear'} kernel SVM, with the hyperparameters returned by the grid search. This model presents high accuracy on the test set, together with a small difference between train and test performance, which indicates good generalization and low overfitting.

Compared to the models from previous tasks (Logistic Regression, LDA/QDA and the tuned Decision Tree / Random Forest), the tuned SVM achieves similar or higher test accuracy while presenting a more stable behaviour across resampling methods. This is consistent with the fact that SVMs, when properly regularized via `C` and `gamma`, often perform very well on high‑dimensional data and imbalanced problems such as the current music classification dataset.


The hyperparameter tuning with 5-fold cross-validation selected a linear SVM with `C = 5` as the best model. This configuration achieved a cross-validation accuracy of 99.96%, a training accuracy of 99.96% and a test accuracy of 100.00%. The very small gap between train and test accuracy indicates that the model does not suffer from overfitting and generalizes extremely well to unseen data.

Compared to the untuned polynomial and RBF kernels, which showed clear signs of overfitting (high training accuracy but significantly lower test accuracy), the tuned linear SVM provides the best balance between bias and variance and is therefore selected as the final SVM model for this task.
